> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 09 — PyTorch for time series: windows, sequence models and conformal intervals

**Sessions:** S17–S18 (PyTorch for financial time series · Sequence models & uncertainty) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Build a window dataset whose targets never leak into the inputs.
2. Train an MLP, an LSTM and a causal TCN, and compare them with a one-line baseline.
3. See what normalizing each window does to a level signal.
4. Wrap any model's forecasts in a split-conformal interval, and watch it fail when volatility changes.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()
import torch
import p10dl as dl

## 1. Windows

Item `i` of a window dataset is the input window `X[i .. i+L−1]` and the target `y[i+L−1]`, where `y` is already the **future** value aligned to the window's last row (here `y[t] = x[t+1]`). With `normalize=True`, each window is scaled with its **own** column means and stds (`(w − w.mean(0)) / (w.std(0) + 1e-8)`): no information from other windows, but also no information about the level. Return two float32 tensors (`torch.from_numpy(w)`, `torch.tensor(...)`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from torch.utils.data import Dataset

class MyWindowDataset(Dataset):
    def __init__(self, X, y, lookback, normalize=True):
        X = np.asarray(X, dtype=np.float32)
        self.X = X.reshape(len(X), -1)                                # (time, features)
        self.y, self.L, self.normalize = np.asarray(y, dtype=np.float32), lookback, normalize

    def __len__(self):
        return len(self.X) - self.L + 1

    def __getitem__(self, i):
        w = self.X[i: i + self.L]
        if self.normalize:
            w = (w - w.mean(0)) / (w.std(0) + 1e-8)
        return torch.from_numpy(w), torch.tensor(self.y[i + self.L - 1])

x = p.ar1(3000, 0.6)                                     # x[t] = 0.6·x[t−1] + noise: the best forecast correlation is 0.6
L = 20
def items(ds):                                           # three windows and their targets, as NumPy arrays
    return [tuple(t.numpy() for t in ds[i]) for i in (0, 7, 2000)]

mine = [p.attempt(items, MyWindowDataset(x[:-1], x[1:], L, norm)) for norm in (True, False)]
mine = p.check("WindowDataset", mine, [items(dl.WindowDataset(x[:-1], x[1:], L, norm)) for norm in (True, False)])
ref_ds = dl.WindowDataset(x[:-1], x[1:], L)
splits = dl.time_splits(len(ref_ds), L)
print(f"{len(ref_ds)} windows → train {len(splits[0])}, validation {len(splits[1])}, test {len(splits[2])}, "
      f"with gaps of {L} windows so no bar is shared")

## 2. Three networks against "tomorrow ∝ today"

`dl.train` is a standard loop: AdamW, gradient clipping, early stopping on the validation loss, best weights restored. Train each model on the same windows, with and without per-window normalization, and compare the test correlation with the naive forecast `x[t]` (for an AR(1) with φ = 0.6, the best possible correlation is about 0.6). About 20 seconds.

In [ ]:
rows = {}
for norm in (True, False):
    ds = dl.WindowDataset(x[:-1], x[1:], L, normalize=norm)
    tr, va, te = dl.loaders(ds, splits)
    for name, make in (("MLP", lambda: dl.MLP(1, L, 16)), ("LSTM", lambda: dl.LSTMRegressor(1, 16)), ("TCN", lambda: dl.TCNRegressor(1, 8, 3))):
        dl.set_seed(0)
        model = make()
        dl.train(model, tr, va, epochs=30, patience=4)
        pred, y_test = dl.predict(model, te)
        rows[(name, "per-window normalized" if norm else "raw windows")] = np.corrcoef(pred, y_test)[0, 1]
naive = x[:-1][splits[2] + L - 1]
rows[("naive: tomorrow ∝ today", "")] = np.corrcoef(naive, y_test)[0, 1]
pd.Series(rows, name="test correlation with the target").round(3).to_frame()

Nothing beats the naive rule, and per-window normalization makes every network worse: it erases the level of `x[t]`, which *is* the signal. The networks learn the AR(1), slowly and with more variance than a one-line rule. That is the normal outcome on simple structure; deep learning must earn its place against baselines like this.

## 3. Split-conformal intervals

Any point forecast can get an interval with a coverage guarantee, if the future looks like the past. On a calibration set (here the validation windows), take the absolute residuals `r`; with `n` of them, `q` is the `k`-th smallest where `k = ceil((n + 1)(1 − α))` (at most `n`); the interval is `pred ± q`. `np.quantile(r, min(1, k/n), method="inverted_cdf")` gives exactly that.

In [ ]:
ds = dl.WindowDataset(x[:-1], x[1:], L, normalize=False)
tr, va, te = dl.loaders(ds, splits)
dl.set_seed(0)
lstm = dl.LSTMRegressor(1, 16)
dl.train(lstm, tr, va, epochs=30, patience=4)
pred_val, y_val = dl.predict(lstm, va)
pred_test, y_test = dl.predict(lstm, te)
res_val = y_val - pred_val

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def conformal_interval(cal_residuals, pred, alpha=0.1):
    r = np.abs(np.asarray(cal_residuals, dtype=float))
    n = len(r)
    q = np.quantile(r, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="inverted_cdf")
    pred = np.asarray(pred, dtype=float)
    return pred - q, pred + q

cases = [(np.arange(1.0, 20.0), np.array([0.0, 10.0]), 0.1), (-np.arange(1.0, 10.0), np.zeros(1), 0.1), (res_val, pred_test, 0.1)]
mine = [p.attempt(conformal_interval, *c) for c in cases]
mine = p.check("conformal_interval", mine, [dl.conformal_interval(*c) for c in cases])
lo, hi = mine[2]
print(f"90% split-conformal interval: ± {(hi - lo)[0] / 2:.3f};  test coverage {dl.coverage(lo, hi, y_test):.3f}")

About 90%, as promised. The promise needs **exchangeability**: calibration and test data from the same distribution. Double the noise in the test period and the same recipe fails:

In [ ]:
rng = np.random.default_rng(9)
noise = rng.normal(0, 1, 3000)
cut = int(0.85 * 3000)
noise[cut:] *= 2                                         # volatility doubles where the test period starts
x2 = np.zeros(3000)
for t in range(1, 3000):
    x2[t] = 0.6 * x2[t - 1] + noise[t]
ds2 = dl.WindowDataset(x2[:-1], x2[1:], L, normalize=False)
tr2, va2, te2 = dl.loaders(ds2, splits)
dl.set_seed(0)
mlp = dl.MLP(1, L, 16)
dl.train(mlp, tr2, va2, epochs=30, patience=4)
pv, yv = dl.predict(mlp, va2)
pt, yt = dl.predict(mlp, te2)
lo2, hi2 = dl.conformal_interval(yv - pv, pt, 0.1)
print(f"calibrated on the calm period, tested after the volatility doubles: coverage {dl.coverage(lo2, hi2, yt):.2f} instead of 0.90")

## Wrap-up

* Windows end at `t`, targets are after `t`; splits need gaps of one look-back.
* Every network gets the same windows as a naive baseline, and usually loses to it on simple structure.
* Don't normalize away the signal: scale with **training** statistics when the level matters.
* Conformal intervals are cheap and honest, until the regime changes: recalibrate on recent data.
* Graded version: `labs/part10/week37_deep` (`WindowDataset`, `time_splits`, `train`, MLP/LSTM/TCN, `conformal_interval`) and Clinic W5 (LSTM vs LightGBM vs HAR).